In [2]:
import tensorflow as tf
from tensorflow import TensorSpec

m = tf.keras.models.load_model("6-Species-Identifier/6_Species_cnn_2025_08_15.keras")
m.trainable = False
for lyr in m.layers:
    if isinstance(lyr, tf.keras.layers.BatchNormalization):
        lyr.trainable = False

@tf.function(input_signature=[TensorSpec([None,256,256,3], tf.float32)])
def serving(x):
    return {"outputs": m(x, training=False)}

tf.saved_model.save(m, "bird_6_saved_v219_infer", signatures={"serving_default": serving})
print("SavedModel -> bird_6_saved_v219_infer")


INFO:tensorflow:Assets written to: bird_6_saved_v219_infer/assets


INFO:tensorflow:Assets written to: bird_6_saved_v219_infer/assets


SavedModel -> bird_6_saved_v219_infer


In [ ]:
# Jupyter/terminal, in TF 2.15.x
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # keep tf.keras v2 route

import tensorflow as tf
print("TF:", tf.__version__)
SM_DIR = "bird_6_saved_v219_infer"   # your TF2.19-exported SavedModel
OUT    = "birds6_tf214_builtin.tflite"

# 1) Load the SavedModel and grab the traced serving fn (already concrete)
sm = tf.saved_model.load(SM_DIR)
concrete = sm.signatures["serving_default"]   # ConcreteFunction

# (Optional) print inputs/outputs to sanity check
print("inputs:", concrete.inputs)
print("outputs:", concrete.outputs)

# 2) Convert FROM CONCRETE FUNCTION (avoids the TF1-style freezing that crashed)
conv = tf.lite.TFLiteConverter.from_concrete_functions([concrete])
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]  # no Select TF ops
conv.optimizations = [tf.lite.Optimize.DEFAULT]
try: conv.experimental_new_converter = False
except: pass
try: conv.experimental_enable_resource_variables = True
except: pass

open(OUT, "wb").write(conv.convert())
print("Wrote:", OUT)
